# Experiment results: Table 2, Changed Knowledge Dataset (512)

Answers are scored against the labels of the question set they answered (see `parse_answers.ipynb`):
- **subset**: `repro/data/medchange_subset.csv`, each question with its own labels. These are the meaningful scores.
- **MedChangeQA.csv**: the authors' released file. Its questions carry another review's labels (see `compare_medchange_datasets.ipynb`), so these rows are only kept for comparison.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

In [2]:
files = {"subset": "output/parsed_labels_subset.csv", "MedChangeQA.csv": "output/parsed_labels.csv"}
dfs = {name: pd.read_csv(path) for name, path in files.items()}

mapper = {"SUPPORTED": 0, "REFUTED": 2, "NOT ENOUGH INFORMATION": 1}
LABEL_COLUMNS = ("Question", "Newest Label", "Outdated Label")
models = {name: [c for c in df.columns if c not in LABEL_COLUMNS] for name, df in dfs.items()}
models, {name: len(df) for name, df in dfs.items()}

({'subset': ['qwen25-7b bf16 (medchange subset)',
   'biomistral-7b (medchange subset)'],
  'MedChangeQA.csv': ['qwen25-7b', 'biomistral-7b (new)']},
 {'subset': 512, 'MedChangeQA.csv': 512})

In [3]:
def macro(y_true, y_pred):
    return (100 * precision_score(y_true, y_pred, average="macro", zero_division=0),
            100 * recall_score(y_true, y_pred, average="macro", zero_division=0),
            100 * f1_score(y_true, y_pred, average="macro", zero_division=0))

rows, index = [], []
for dataset, df in dfs.items():
    y_out = df["Outdated Label"].map(mapper).to_numpy()
    y_new = df["Newest Label"].map(mapper).to_numpy()
    for m in models[dataset]:
        y_pred = df[m].map(mapper).to_numpy()
        po, ro, fo = macro(y_out, y_pred)
        pn, rn, fn = macro(y_new, y_pred)
        rows.append({
            ("Outdated Lab.", "P"): po, ("Outdated Lab.", "R"): ro, ("Outdated Lab.", "F1"): fo,
            ("Latest Labels", "P"): pn, ("Latest Labels", "R"): rn, ("Latest Labels", "F1"): fn,
            ("", "F1 diff."): fn - fo,
            ("", "Outdated Answers"): 100 * np.mean(y_pred == y_out),
        })
        index.append((dataset, m))

# Paper Table 2 for reference (F1 diff. as printed; BioMistral's is -1.0 by arithmetic)
paper = {
    "qwen25-7b":     (42.6, 37.1, 30.8, 27.1, 30.8, 26.0, -4.8, 35.4),
    "biomistral-7b": (36.8, 37.2, 36.3, 35.4, 35.5, 35.3, -1.5, 37.1),
}
for m, values in paper.items():
    rows.append(dict(zip(rows[0].keys(), values)))
    index.append(("paper Table 2", m))

table = pd.DataFrame(rows, index=pd.MultiIndex.from_tuples(index, names=["questions", "model"]))
table.columns = pd.MultiIndex.from_tuples(table.columns)
table.round(1)

Outdated Lab.              \
                                                              P     R    F1   
questions       model                                                         
subset          qwen25-7b bf16 (medchange subset)          43.1  37.2  30.9   
                biomistral-7b (medchange subset)           24.1  32.0  26.9   
MedChangeQA.csv qwen25-7b                                  30.4  33.5  26.1   
                biomistral-7b (new)                        24.6  32.3  27.5   
paper Table 2   qwen25-7b                                  42.6  37.1  30.8   
                biomistral-7b                              36.8  37.2  36.3   

                                                  Latest Labels              \
                                                              P     R    F1   
questions       model                                                         
subset          qwen25-7b bf16 (medchange subset)          28.3  30.7  26.0   
                biomistral-7b (medchange subset)           25.9  34.8  29.7   
MedChangeQA.csv qwen25-7b                                  34.9  32.1  28.1   
                biomistral-7b (new)                        24.9  33.6  28.5   
paper Table 2   qwen25-7b                                  27.1  30.8  26.0   
                biomistral-7b                              35.4  35.5  35.3   

                                                                             
                                                  F1 diff. Outdated Answers  
questions       model                                                        
subset          qwen25-7b bf16 (medchange subset)     -4.9             35.9  
                biomistral-7b (medchange subset)       2.8             35.5  
MedChangeQA.csv qwen25-7b                              1.9             33.8  
                biomistral-7b (new)                    0.9             36.7  
paper Table 2   qwen25-7b                             -4.8             35.4  
                biomistral-7b                         -1.5             37.1

In [4]:
# formatted like the paper (bold = best per column is not marked here; diff has a sign, answers a %)
fmt = table.copy()
for c in fmt.columns:
    fmt[c] = fmt[c].map(("{:+.1f}" if c[1] == "F1 diff." else "{:.1f}%" if c[1] == "Outdated Answers" else "{:.1f}").format)
fmt

Outdated Lab.              \
                                                              P     R    F1   
questions       model                                                         
subset          qwen25-7b bf16 (medchange subset)          43.1  37.2  30.9   
                biomistral-7b (medchange subset)           24.1  32.0  26.9   
MedChangeQA.csv qwen25-7b                                  30.4  33.5  26.1   
                biomistral-7b (new)                        24.6  32.3  27.5   
paper Table 2   qwen25-7b                                  42.6  37.1  30.8   
                biomistral-7b                              36.8  37.2  36.3   

                                                  Latest Labels              \
                                                              P     R    F1   
questions       model                                                         
subset          qwen25-7b bf16 (medchange subset)          28.3  30.7  26.0   
                biomistral-7b (medchange subset)           25.9  34.8  29.7   
MedChangeQA.csv qwen25-7b                                  34.9  32.1  28.1   
                biomistral-7b (new)                        24.9  33.6  28.5   
paper Table 2   qwen25-7b                                  27.1  30.8  26.0   
                biomistral-7b                              35.4  35.5  35.3   

                                                                             
                                                  F1 diff. Outdated Answers  
questions       model                                                        
subset          qwen25-7b bf16 (medchange subset)     -4.9            35.9%  
                biomistral-7b (medchange subset)      +2.8            35.5%  
MedChangeQA.csv qwen25-7b                             +1.9            33.8%  
                biomistral-7b (new)                   +0.9            36.7%  
paper Table 2   qwen25-7b                             -4.8            35.4%  
                biomistral-7b                         -1.5            37.1%

In [5]:
# sanity check: label distribution of gold vs predictions, per question set
for dataset, df in dfs.items():
    print(dataset)
    display(pd.concat({"Outdated gold": df["Outdated Label"].value_counts(),
                       "Newest gold": df["Newest Label"].value_counts(),
                       **{m: df[m].value_counts() for m in models[dataset]}}, axis=1).fillna(0).astype(int))

subset


,Outdated gold,Newest gold,qwen25-7b bf16 (medchange subset),biomistral-7b (medchange subset)
NOT ENOUGH INFORMATION,237,160,72,235
SUPPORTED,152,221,394,277
REFUTED,123,131,46,0


MedChangeQA.csv


,Outdated gold,Newest gold,qwen25-7b,biomistral-7b (new)
NOT ENOUGH INFORMATION,237,160,114,251
SUPPORTED,152,221,379,261
REFUTED,123,131,19,0


In [6]:
table.round(1).to_csv("output/table2_changed_knowledge.csv")